## 0. Импорт

In [130]:
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

## 1. Предварительная обработка
1) Создайте тот же датафрейм, что и в предыдущем упражнении.
2) Используя train_test_splitпараметры test_size=0.2, random_state=21получите X_train, y_train, X_test, y_testа затем получите X_train, y_train, X_valid, y_validиз предыдущего X_train, y_train. Используйте дополнительный параметр stratify.


In [131]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
lastDf = pd.read_csv('../data/dayofweek.csv')
df = lastDf[['dayofweek'] + [c for c in df if c != 'dayofweek']]
df.head(3)

,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [132]:
X = df[df.drop(columns='dayofweek').columns]
y = df['dayofweek']
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=21, stratify=y_temp
)

## 2. Отдельные классификаторы
1) Обучите SVM, дерево решений и случайный лес еще раз, используя лучшие параметры, полученные в упражнении 01, random_state=21для всех этих алгоритмов.
2) Оцените accuracy, precision, и recallдля них на проверочном наборе данных.
3) Результат обработки каждой ячейки раздела должен выглядеть следующим образом: 
```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778

In [133]:
svc = SVC( C= 5, class_weight = None, gamma = 'scale', kernel = 'rbf', probability=True, random_state=21)
tree = DecisionTreeClassifier(class_weight = None, criterion = 'gini', max_depth = 16,random_state=21)
rndForest = joblib.load('../models/bestModel_rndForest.joblib')


In [134]:
def models_metrics(models, params, X_train, X_test, y_train, y_test):
    results = {}
    for model, params in zip(models, params):
        model.set_params(**params)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        
    
        results[model.__class__.__name__] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
        }
    return results

In [135]:
models_to_test = [
    svc, tree, rndForest
]

params_to_test = [
    {}, 
    {},                           
    {}    
]

metrics_dict = models_metrics(models_to_test, params_to_test, X_train,  X_test, y_train, y_test)

for model_name, metrics in metrics_dict.items():
    print(f"Model: {model_name}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name} is {value:.5f}")
    print()

Model: SVC
  accuracy is 0.86982
  precision is 0.87227
  recall is 0.86982

Model: DecisionTreeClassifier
  accuracy is 0.85207
  precision is 0.85440
  recall is 0.85207

Model: RandomForestClassifier
  accuracy is 0.88462
  precision is 0.88821
  recall is 0.88462



## 3. Классификаторы голосования
1) Используя VotingClassifier три модели, которые вы только что обучили, рассчитайте значения accuracy, precision, и recallна валидационном наборе данных.
2) Поэкспериментируйте с другими параметрами.
3) Рассчитайте значения accuracy, precisionи recallна тестовом наборе для модели с наилучшими весами с точки зрения точности (если есть несколько моделей с одинаковыми значениями, выберите ту, у которой более высокая точность).

In [136]:
enable = VotingClassifier(
    estimators=[('svc', svc), ('tree', tree), ('rndForest', rndForest)], 
    voting='hard'
)

enable.fit(X_train, y_train)

y_val_pred = enable.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, average='weighted')
recall = recall_score(y_val, y_val_pred, average='weighted')

print(f"Accuracy: {acc:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.86944
Precision: 0.86906
Recall: 0.86944


In [137]:
enable = VotingClassifier(
    estimators=[('svc', svc), ('tree', tree), ('rndForest', rndForest)], 
    voting='soft'
)

enable.fit(X_train, y_train)

y_val_pred = enable.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, average='weighted')
recall = recall_score(y_val, y_val_pred, average='weighted')

print(f"Accuracy: {acc:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.86944
Precision: 0.87137
Recall: 0.86944


In [138]:
best_weights = [0,0,1]
enable = VotingClassifier(
    estimators=[('svc', svc), ('tree', tree), ('rndForest', rndForest)], 
    voting='soft',
    weights = best_weights
)

enable.fit(X_train, y_train)

y_pred = enable.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {acc:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.88462
Precision: 0.88821
Recall: 0.88462


## 4. Классификаторы бэггинга
1) Используя BaggingClassifier и SVM с наилучшими параметрами, создайте ансамбль, попробуйте разные значения n_estimators, используйте random_state=21.
2) Поэкспериментируйте с другими параметрами.
3) Рассчитайте значения accuracy, precision, и recallдля модели с наилучшими параметрами (с точки зрения точности) на тестовом наборе данных (если есть несколько моделей с одинаковыми значениями, выберите ту, у которой более высокая точность).

In [139]:
bagging = BaggingClassifier(estimator=svc, n_estimators=7, random_state=21)

bagging.fit(X_train, y_train)

y_val_pred = bagging.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, average='weighted')
recall = recall_score(y_val, y_val_pred, average='weighted')

print(f"Accuracy: {acc:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.84866
Precision: 0.85105
Recall: 0.84866


In [140]:
y_pred = bagging.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {acc:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.87574
Precision: 0.88026
Recall: 0.87574


## 5. Классификаторы стекирования
1) Для обеспечения воспроизводимости в данном случае вам потребуется создать объект генератора перекрестной проверки: StratifiedKFold(n_splits=n, shuffle=True, random_state=21), где n вы попытаетесь оптимизировать (подробности ниже).
2) Используя StackingClassifier три недавно обученные модели, рассчитайте значение accuracy, precisionа recall на проверочном наборе данных попробуйте разные значения n_splits [2, 3, 4, 5, 6, 7] в генераторе перекрестной проверки и параметра passthrough в самом классификаторе.
3) Рассчитайте accuracy, precision, и recall для модели с наилучшими параметрами (с точки зрения точности) на тестовом наборе данных (если есть несколько моделей с одинаковыми значениями, выберите ту, у которой более высокая точность). Используйте final_estimator=LogisticRegression(solver='liblinear').

In [141]:
final_estimator = OneVsRestClassifier(
    LogisticRegression(solver='liblinear', random_state=21)
)
estimators = [('svc', svc), ('tree', tree), ('rndForest', rndForest)]
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [False, True]

results = []
best_acc = -1.0
best_prec = -1.0
best_params = {}

for n in n_splits_list:
    for pt in passthrough_list:
        cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
        
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=final_estimator,
            cv=cv,
            passthrough=pt,
            n_jobs=-1 
        )
        
        stacking_clf.fit(X_train, y_train)
        
        y_val_pred = stacking_clf.predict(X_val)
        
        acc = accuracy_score(y_val, y_val_pred)
        prec = precision_score(y_val, y_val_pred, zero_division=0,average='weighted')
        rec = recall_score(y_val, y_val_pred, zero_division=0,average='weighted')
        
        results.append({
            'n_splits': n,
            'passthrough': pt,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec
        })
        if (acc > best_acc) or (acc == best_acc and prec > best_prec):
            best_acc = acc
            best_prec = prec
            best_params = {'n_splits': n, 'passthrough': pt}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nbest params: n_splits={best_params['n_splits']}, passthrough={best_params['passthrough']}")

cv_best = StratifiedKFold(n_splits=best_params['n_splits'], shuffle=True, random_state=21)

best_model = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=cv_best,
    passthrough=best_params['passthrough'],
    n_jobs=-1
)

best_model.fit(X_train, y_train)

y_test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_rec = recall_score(y_test, y_test_pred, zero_division=0, average='weighted')


print(f"Accuracy:  {test_acc:.5f}")
print(f"Precision: {test_prec:.5f}")
print(f"Recall:    {test_rec:.5f}")

 n_splits  passthrough  Accuracy  Precision   Recall
        2        False  0.869436   0.870396 0.869436
        2         True  0.875371   0.876142 0.875371
        3        False  0.863501   0.864535 0.863501
        3         True  0.869436   0.871695 0.869436
        4        False  0.875371   0.877038 0.875371
        4         True  0.875371   0.876457 0.875371
        5        False  0.875371   0.877449 0.875371
        5         True  0.884273   0.885147 0.884273
        6        False  0.869436   0.871282 0.869436
        6         True  0.881306   0.882714 0.881306
        7        False  0.878338   0.880130 0.878338
        7         True  0.884273   0.886297 0.884273

best params: n_splits=7, passthrough=True
Accuracy:  0.89349
Precision: 0.89629
Recall:    0.89349


## 6. Прогнозы
1) Выберите лучшую модель с точки зрения точности (если есть несколько моделей с одинаковыми значениями, выберите ту, у которой более высокая точность).
2) Проанализируйте: для какого дня недели ваша модель допускает наибольшее количество ошибок (в процентах от общего числа образцов этого класса в вашем полном наборе данных), для какого названия лаборатории и для каких пользователей.
3) Сохраните модель.

In [142]:
best_model = best_model ## имею ввиду модель с прошлого пункта

df_test = X_test.copy()
df_test['true_label'] = y_test
df_test['pred_label'] = best_model.predict(X_test)

errors = df_test[df_test['true_label'] != df_test['pred_label']]
errors_count = errors['true_label'].value_counts()
total_count = df_test['true_label'].value_counts()

errors_percentage = (errors_count/total_count).sort_values(ascending=False)

users = [user for user in errors.columns if user.startswith('uid_user_')]
errors_labs = errors.drop(columns=['true_label', 'pred_label']+users).sum().sort_values(ascending=False)

labs = [lab for lab in errors.columns if lab.startswith('labname_') ]
errors_users = errors.drop(columns=['true_label', 'pred_label']+labs).sum().sort_values(ascending=False)

print("\nТоп классов с наибольшим % ошибок:")
print(errors_percentage.head())
print("\nТоп лабораторных с наибольшим % ошибок:")
print(errors_labs.head())
print("\nТоп пользователей с наибольшим % ошибок:")
print(errors_users.head())


Топ классов с наибольшим % ошибок:
true_label
0    0.259259
4    0.142857
5    0.111111
1    0.109091
2    0.100000
Name: count, dtype: float64

Топ лабораторных с наибольшим % ошибок:
labname_project1    16.0
labname_laba04      10.0
labname_laba04s      4.0
labname_laba06s      2.0
labname_code_rvw     1.0
dtype: float64

Топ пользователей с наибольшим % ошибок:
uid_user_13    5.0
uid_user_3     4.0
uid_user_4     3.0
uid_user_14    3.0
uid_user_2     3.0
dtype: float64


In [144]:
joblib.dump(best_model, '../models/SKF_best.joblib')

['../models/SKF_best.joblib']